In [14]:
import pandas as pd
import h3

In [6]:
wac = pd.read_csv('./data/wac_929fc6875af39f9b861b90560a08aa7c.csv')
xw = pd.read_csv('./data/xwalk_929fc6875af39f9b861b90560a08aa7c.csv')

/var/folders/1_/cybvzsy149s827970j9wcvh40000gn/T/ipykernel_10714/1854909005.py:2: DtypeWarning: Columns (0: milname) have mixed types. Specify dtype option on import or set low_memory=False.
  xw = pd.read_csv('./data/xwalk_929fc6875af39f9b861b90560a08aa7c.csv')


In [7]:
wac.head()

,w_geocode,segment,year,C000
0,360050033001002,S000,2023,65
1,360050033001004,S000,2023,35
2,360050033001005,S000,2023,148
3,360050033001006,S000,2023,3
4,360050033002000,S000,2023,98


In [8]:
xw.head()

,tabblk2020,st,stusps,stname,cty,ctyname,trct,trctname,bgrp,bgrpname,...,tsub,tsubname,stanrc,stanrcname,mil,milname,stwib,stwibname,blklatdd,blklondd
0,360050033002000,36,NY,New York,36005,"Bronx County, NY",36005003300,"33 (Bronx, NY)",360050033002,"2 (Tract 33, Bronx, NY)",...,NaN,NaN,NaN,NaN,NaN,NaN,36360151,New York City LWIA,40.808316,-73.912493
1,360050033002001,36,NY,New York,36005,"Bronx County, NY",36005003300,"33 (Bronx, NY)",360050033002,"2 (Tract 33, Bronx, NY)",...,NaN,NaN,NaN,NaN,NaN,NaN,36360151,New York City LWIA,40.808635,-73.913389
2,360050042003000,36,NY,New York,36005,"Bronx County, NY",36005004200,"42 (Bronx, NY)",360050042003,"3 (Tract 42, Bronx, NY)",...,NaN,NaN,NaN,NaN,NaN,NaN,36360151,New York City LWIA,40.825946,-73.861150
3,360050042003001,36,NY,New York,36005,"Bronx County, NY",36005004200,"42 (Bronx, NY)",360050042003,"3 (Tract 42, Bronx, NY)",...,NaN,NaN,NaN,NaN,NaN,NaN,36360151,New York City LWIA,40.825755,-73.863213
4,360050042003002,36,NY,New York,36005,"Bronx County, NY",36005004200,"42 (Bronx, NY)",360050042003,"3 (Tract 42, Bronx, NY)",...,NaN,NaN,NaN,NaN,NaN,NaN,36360151,New York City LWIA,40.824999,-73.868815


In [9]:
wac = wac[wac['segment'] == 'S000'][['w_geocode', 'C000']]
xw = xw[['tabblk2020', 'cty', 'blklatdd', 'blklondd']]

# Filter xwalk to NYC counties only
nyc_counties = [36005, 36047, 36061, 36081, 36085]
xw = xw[xw['cty'].isin(nyc_counties)]

In [16]:
# Join jobs to block coordinates
df = wac.merge(xw, left_on='w_geocode', right_on='tabblk2020', how='inner')

# Assign each block centroid to H3 cell at each resolution
for res in [7, 8, 9]:
    df[f'h3_res{res}'] = df.apply(
        lambda row: h3.latlng_to_cell(row['blklatdd'], row['blklondd'], res), axis=1
    )

df

,w_geocode,C000,tabblk2020,cty,blklatdd,blklondd,h3_res7,h3_res8,h3_res9
0,360050033001002,65,360050033001002,36005,40.808304,-73.910439,872a100f6ffffff,882a100f69fffff,892a100f69bffff
1,360050033001004,35,360050033001004,36005,40.807636,-73.911823,872a100f6ffffff,882a100f6dfffff,892a100f6d7ffff
2,360050033001005,148,360050033001005,36005,40.807367,-73.910944,872a100f6ffffff,882a100f69fffff,892a100f6d7ffff
3,360050033001006,3,360050033001006,36005,40.806936,-73.909492,872a100f6ffffff,882a100f69fffff,892a100f69bffff
4,360050033002000,98,360050033002000,36005,40.808316,-73.912493,872a100f6ffffff,882a100f6dfffff,892a100f6d7ffff
...,...,...,...,...,...,...,...,...,...
25440,360850323001017,5,360850323001017,36085,40.627405,-74.168705,872a10622ffffff,882a106225fffff,892a1062257ffff
25441,360850323001018,20,360850323001018,36085,40.629380,-74.168821,872a10622ffffff,882a106225fffff,892a1062257ffff
25442,360850323001021,39,360850323001021,36085,40.626397,-74.172910,872a10622ffffff,882a106225fffff,892a1062247ffff
25443,360850323001022,65,360850323001022,36085,40.625202,-74.170621,872a10622ffffff,882a106225fffff,892a1062247ffff


In [17]:
H3_AREA_KM2 = {7: 5.1613, 8: 0.7373, 9: 0.1053}

for res in [7, 8, 9]:
    out = (
        df.groupby(f'h3_res{res}')['C000']
        .sum()
        .reset_index()
        .rename(columns={f'h3_res{res}': 'h3_index', 'C000': 'jobs'})
    )
    out['density'] = (out['jobs'] / H3_AREA_KM2[res]).round(2)
    out['jobs'] = out['jobs'].round(0).astype(int)
    print(f"Res {res}: {len(out)} cells, {out['jobs'].sum():,} total jobs")
    # push_to_supabase(out, f"h3_jobs_res{res}")

Res 7: 200 cells, 4,585,129 total jobs
Res 8: 1094 cells, 4,585,129 total jobs
Res 9: 5902 cells, 4,585,129 total jobs


In [ ]:
# SANITY CHECK

import geopandas as gpd
from shapely.geometry import Polygon
import matplotlib.pyplot as plt

for res in [7, 8, 9]:
    out = (
        df.groupby(f'h3_res{res}')['C000']
        .sum()
        .reset_index()
        .rename(columns={f'h3_res{res}': 'h3_index', 'C000': 'jobs'})
    )
    
    hex_geoms = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in out['h3_index']]
    gdf = gpd.GeoDataFrame(out, geometry=hex_geoms, crs='EPSG:4326')
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 12))
    gdf.plot(column='jobs', ax=ax, cmap='YlOrRd', legend=True,
             legend_kwds={'label': 'Jobs per H3 Cell', 'shrink': 0.6})
    ax.set_title(f'NYC Jobs by H3 Cell - Res {res} (2023)', fontsize=16)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

***MTA RIDERSHIP***

In [25]:
import requests

def fetch_mta_ridership():
    print("Fetching MTA ridership data...")
    
    # Socrata API — filter to 2024 weekdays only, aggregate by station
    # $select aggregates server-side so we only download the summary
    url = (
        "https://data.ny.gov/resource/wujg-7c2s.json"
        "?$select=station_complex,latitude,longitude,sum(ridership) as total_ridership"
        "&$where=transit_timestamp >= '2024-01-01' AND transit_timestamp < '2024-04-01'"
        "&$group=station_complex,latitude,longitude"
        "&$limit=1000"
    )
    
    r = requests.get(url)
    df = pd.DataFrame(r.json())
    df['total_ridership'] = pd.to_numeric(df['total_ridership'])
    df['latitude'] = pd.to_numeric(df['latitude'])
    df['longitude'] = pd.to_numeric(df['longitude'])
    
    print(f"  Fetched {len(df)} stations, {df['total_ridership'].sum():,.0f} total rides")
    return df

In [28]:
fetch_mta_ridership().to_csv('./data/mta_ridership_q1_2024.csv', index=False)

Fetching MTA ridership data...
  Fetched 488 stations, 286,145,962 total rides


In [29]:
mta = pd.read_csv('./data/mta_ridership_q1_2024.csv')

for res in [7, 8, 9]:
    print(f"\nProcessing MTA H3 resolution {res}...")
    mta[f'h3_res{res}'] = mta.apply(
        lambda row: h3.latlng_to_cell(row['latitude'], row['longitude'], res), axis=1
    )
    out = (
        mta.groupby(f'h3_res{res}')['total_ridership']
        .sum()
        .reset_index()
        .rename(columns={f'h3_res{res}': 'h3_index', 'total_ridership': 'ridership'})
    )
    out['ridership'] = out['ridership'].round(0).astype(int)
    print(f"  Cells: {len(out)}, Total ridership: {out['ridership'].sum():,.0f}")
    # push_to_supabase(out, f"h3_transit_res{res}")


Processing MTA H3 resolution 7...
  Cells: 89, Total ridership: 286,145,962

Processing MTA H3 resolution 8...
  Cells: 281, Total ridership: 286,145,962

Processing MTA H3 resolution 9...
  Cells: 426, Total ridership: 286,145,962
